In [5]:
import pandas as pd
import plotly.graph_objects as go

# Path to the main Sankey data
excel_file_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\Fraunhofer\combined_summary_full.xlsx'

# Path to the material composition data
composition_file_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\Fraunhofer\combined_excel_material_composition.xlsx'

# Load the main Sankey data
df = pd.read_excel(excel_file_path)
df = df.fillna(method='ffill')

# Load the material composition data
composition_df = pd.read_excel(composition_file_path)

# Extract values from the first row (row index 0)
node_label_1 = df.iloc[0, 0]  # Value at position (0,0)
node_label_2 = df.iloc[0, 1]  # Value at position (0,1)
link_value = df.iloc[0, 2]    # Value at position (0,2)

# Create the nodes part
nodes = [
    {'label': node_label_1},
    {'label': node_label_2}
]

# Create the link part
links = [
    {
        'source': node_label_1,
        'target': node_label_2,
        'value': link_value
    }
]

df = df.drop(columns=['Process Name', 'Input Material Amount'])

node_labels = {node_label_1, node_label_2}

for idx, row in df.iterrows():
    src = row[0]  # First column as source
    tgt = row[1]  # Second column as target
    val = row[2]  # Third column as link value

    # Check and add the source node if not already present
    if src not in node_labels:
        nodes.append({'label': src})
        node_labels.add(src)

    # Check and add the target node if not already present
    if tgt not in node_labels:
        nodes.append({'label': tgt})
        node_labels.add(tgt)

    # Append the link from source to target with the given value
    links.append({
        'source': src,
        'target': tgt,
        'value': val
    })

# Define the list of nodes for which material composition needs to be added
nodes_with_composition = ["Plastics in Different DS", "Mixed Plastics", "PP in PP sorted products", "PS in PS sorted products", "PET in PET sorted products", "HDPE in HDPE sorted products", "Film in film sorted products"]  # Replace with actual node names

for node_name in nodes_with_composition:
    # Filter the material composition data for this specific node
    filtered_composition_df = composition_df[composition_df['Material name'] == node_name]

    # Create a dictionary with 'Name of the Materials' as keys and 'Amount' as values
    material_composition = dict(zip(filtered_composition_df['Name of the Materials'], filtered_composition_df['Amount']))

    # Add the node if it is not already present
    if node_name not in node_labels:
        nodes.append({'label': node_name})
        node_labels.add(node_name)

    # Add material composition nodes and dashed links
    for material, value in material_composition.items():
        if material not in node_labels:
            nodes.append({'label': material})
            node_labels.add(material)

        # Add dashed-like links with minimal values and unique color
        links.append({
            'source': node_name,  # Dynamic node as the source
            'target': material,   # Material as the target
            'value': 0.01         # Minimal value for dashed effect
        })

# Prepare Data for Plotly Sankey
node_labels = [node['label'] for node in nodes]
label_to_index = {label: idx for idx, label in enumerate(node_labels)}

source_indices = [label_to_index[link['source']] for link in links]
target_indices = [label_to_index[link['target']] for link in links]
values = [link['value'] for link in links]

# Assign unique colors for dashed or transparent links
link_colors = []
for link in links:
    if link['value'] == 0.01:  # Dashed-like links (composition)
        link_colors.append("rgba(0, 0, 0, 0.2)")  # Light grey for dashed effect
    else:  # Regular flow links
        link_colors.append("rgba(44, 160, 44, 0.8)")  # Green for regular links

# Create the Sankey Diagram with Plotly
fig = go.Figure(data=[go.Sankey(
    arrangement="freeform",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=values,
        color=link_colors
    )
)])

# Add layout details
fig.update_layout(title_text="Plastic Waste Management - Sankey Diagram", font_size=12)

# Show the diagram
fig.show()
